# Advanced Module 2 · Agentic Loops & ReAct
Agents that reason in loops — from a bare ReAct loop to self-correcting multi-step agents.

---
> **Prerequisite:** Complete Module 1 (Function Calling & Tools) first.  
> **Setup:** Run the first two cells before anything else.

In [ ]:
%pip install -q google-genai groq python-dotenv

In [8]:
import os, json, math, time, base64, getpass, re
from dotenv import load_dotenv
from google import genai
from google.genai import types
from groq import Groq

load_dotenv()
api_key = os.environ.get('GEMINI_API_KEY')
if not api_key:
    api_key = getpass.getpass('Paste your Gemini API key: ')

groq_key = os.environ.get('Groq_key', '').strip()
if not groq_key:
    groq_key = getpass.getpass('Paste your Groq API key: ')

client      = genai.Client(api_key=api_key)
groq_client = Groq(api_key=groq_key)
MODEL       = 'gemini-3.1-flash-lite'
GROQ_MODEL  = 'llama-3.3-70b-versatile'
DEFAULT_MAX = 2048

def cfg(**kwargs):
    kwargs.setdefault('max_output_tokens', DEFAULT_MAX)
    kwargs.setdefault('temperature', 0.7)
    return types.GenerateContentConfig(**kwargs)

def get_text(response) -> str:
    if response.text:
        return response.text.strip()
    parts = []
    for cand in (response.candidates or []):
        if cand.content:
            for part in cand.content.parts:
                if not getattr(part, 'thought', False) and part.text:
                    parts.append(part.text)
    return ''.join(parts).strip()

def _call_with_retry(fn, *args, max_retries=5, **kwargs):
    for attempt in range(max_retries):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            msg = str(e)
            if '429' in msg or 'RESOURCE_EXHAUSTED' in msg:
                m = re.search(r'retry[^0-9]*([0-9]+)s', msg, re.I)
                wait = int(m.group(1)) + 5 if m else 35
                print(f'  ⏳ Rate-limited — waiting {wait}s (attempt {attempt+1}/{max_retries})')
                time.sleep(wait)
            elif '500' in msg or 'INTERNAL' in msg:
                wait = 10 * (attempt + 1)
                print(f'  ⏳ Server error — waiting {wait}s (attempt {attempt+1}/{max_retries})')
                time.sleep(wait)
            else:
                raise
    raise RuntimeError('Max retries exceeded')

_raw_gen = client.models.generate_content
client.models.generate_content = lambda *a, **kw: _call_with_retry(_raw_gen, *a, **kw)

print('✅ Setup complete. Gemini:', MODEL, '| Groq:', GROQ_MODEL)

✅ Setup complete. Gemini: gemini-3.1-flash-lite | Groq: llama-3.3-70b-versatile


In [9]:
try:
    _r = client.models.generate_content(model=MODEL,
        contents='Reply with exactly the words: Hello LLM course', config=cfg(temperature=0.0))
    print('✅ Gemini API working:', get_text(_r))
except Exception as e:
    print('❌ Gemini error:', e)

✅ Gemini API working: Hello LLM course


---
## The Core Idea: An Agent Is Just a While-Loop

In Module 1 we saw single and chained tool calls. An **agent** takes this further:  
it loops — calling tools, observing results, reasoning about them — until the task is done.

**ReAct** (Reason + Act) is the most popular pattern: the model alternates between  
writing a **Thought** (reasoning) and taking an **Action** (tool call), then observing the result.

```
┌─────────────────────────────────────────────────────────────────┐
│                     THE REACT AGENT LOOP                        │
│                                                                 │
│   User task: "Book cheapest flight + 3-night hotel"             │
│          │                                                      │
│          ▼                                                      │
│   ┌─────────────┐                                               │
│   │  THOUGHT 1  │  "I need to search for flights first"         │
│   └──────┬──────┘                                               │
│          ▼                                                      │
│   ┌─────────────┐                                               │
│   │  ACTION 1   │  search_flights(origin='BOM', dest='BER')     │
│   └──────┬──────┘                                               │
│          ▼                                                      │
│   ┌─────────────┐                                               │
│   │ OBSERVATION │  [{'id':'F1','price':45000}, ...]             │
│   └──────┬──────┘                                               │
│          ▼                                                      │
│   ┌─────────────┐                                               │
│   │  THOUGHT 2  │  "F1 is cheapest. Now I need a hotel"         │
│   └──────┬──────┘                                               │
│          ▼                                                      │
│   ┌─────────────┐                                               │
│   │  ACTION 2   │  search_hotels(city='Berlin', nights=3)       │
│   └──────┬──────┘                                               │
│          ▼                                                      │
│   ┌─────────────┐                                               │
│   │ OBSERVATION │  [{'id':'H2','price_per_night':8500}, ...]    │
│   └──────┬──────┘                                               │
│          ▼                                                      │
│   ┌─────────────┐                                               │
│   │ FINAL ANSWER│  "Cheapest option: F1 + H2 = ₹70,500 total"  │
│   └─────────────┘                                               │
└─────────────────────────────────────────────────────────────────┘
```

Key difference from Module 1: **the agent decides dynamically** which tools to call and in what order.  
No hardcoded sequence — just reasoning.

---
## Demo 1: Bare ReAct Loop from Scratch

No frameworks. A pure Python while-loop. Every Thought and Action printed explicitly.

In [10]:
# Stub tool library
def get_weather(city: str) -> dict:
    weather_db = {
        'mumbai':  {'temp': 32, 'condition': 'humid', 'rain_chance': 80},
        'berlin':  {'temp': 18, 'condition': 'cloudy', 'rain_chance': 30},
        'london':  {'temp': 15, 'condition': 'rainy',  'rain_chance': 75},
        'dubai':   {'temp': 42, 'condition': 'sunny',  'rain_chance': 2},
        'toronto': {'temp': 22, 'condition': 'clear',  'rain_chance': 10},
    }
    return weather_db.get(city.lower(), {'temp': 20, 'condition': 'unknown', 'rain_chance': 50})

def convert_currency(amount: float, from_currency: str, to_currency: str) -> dict:
    rates = {'USD': 1.0, 'EUR': 0.93, 'GBP': 0.79, 'INR': 83.5, 'AED': 3.67}
    base  = amount / rates.get(from_currency.upper(), 1.0)
    converted = round(base * rates.get(to_currency.upper(), 1.0), 2)
    return {'from': f'{amount} {from_currency}', 'to': f'{converted} {to_currency}'}

def get_flight_info(origin: str, destination: str) -> dict:
    return {
        'route': f'{origin} → {destination}',
        'duration_hours': 9,
        'cheapest_fare_usd': 620,
        'airline': 'SkyWings'
    }

TOOLS = {
    'get_weather':       get_weather,
    'convert_currency':  convert_currency,
    'get_flight_info':   get_flight_info,
}

react_tool_schema = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name='get_weather',
            description='Get current weather for a city.',
            parameters=types.Schema(type=types.Type.OBJECT,
                properties={'city': types.Schema(type=types.Type.STRING)}, required=['city'])
        ),
        types.FunctionDeclaration(
            name='convert_currency',
            description='Convert an amount from one currency to another.',
            parameters=types.Schema(type=types.Type.OBJECT,
                properties={
                    'amount':        types.Schema(type=types.Type.NUMBER),
                    'from_currency': types.Schema(type=types.Type.STRING, description='e.g. USD, EUR, INR'),
                    'to_currency':   types.Schema(type=types.Type.STRING, description='e.g. USD, EUR, INR'),
                },
                required=['amount', 'from_currency', 'to_currency'])
        ),
        types.FunctionDeclaration(
            name='get_flight_info',
            description='Get flight information between two cities.',
            parameters=types.Schema(type=types.Type.OBJECT,
                properties={
                    'origin':      types.Schema(type=types.Type.STRING),
                    'destination': types.Schema(type=types.Type.STRING),
                },
                required=['origin', 'destination'])
        ),
    ]
)

print('✅ Tools ready:', list(TOOLS.keys()))

✅ Tools ready: ['get_weather', 'convert_currency', 'get_flight_info']


In [11]:
def run_react_agent(task: str, max_steps: int = 8, verbose: bool = True) -> str:
    """Pure Python ReAct loop — no frameworks."""
    conversation  = [types.Content(role='user', parts=[types.Part(text=task)])]
    total_tokens  = 0
    step          = 0

    if verbose:
        print('=' * 65)
        print(f'TASK: {task}')
        print('=' * 65)

    while step < max_steps:
        step += 1
        response = client.models.generate_content(
            model=MODEL, contents=conversation,
            config=cfg(tools=[react_tool_schema], temperature=0.0)
        )
        total_tokens += response.usage_metadata.total_token_count

        # Collect all function calls from this response
        fcs = [
            p.function_call
            for p in response.candidates[0].content.parts
            if hasattr(p, 'function_call') and p.function_call
        ]

        if not fcs:
            # No tool call = final answer
            final = get_text(response)
            if verbose:
                print(f'\n[Step {step}] THOUGHT → Final answer (no more tools needed)')
                print(f'[Step {step}] ANSWER  → {final}')
                print(f'\n📊 Total steps: {step} | Total tokens used: {total_tokens}')
            return final

        # Execute tools and build observation
        results = []
        for fc in fcs:
            result = TOOLS[fc.name](**dict(fc.args))
            results.append(result)
            if verbose:
                print(f'[Step {step}] ACTION      → {fc.name}({dict(fc.args)})')
                print(f'[Step {step}] OBSERVATION → {result}')

        # Add to conversation
        conversation.append(response.candidates[0].content)  # preserves thought_signature
        conversation.append(types.Content(role='user', parts=[
            types.Part(function_response=types.FunctionResponse(name=fc.name, response={'result': r}))
            for fc, r in zip(fcs, results)
        ]))

    return 'Max steps reached without a final answer.'

# Run it
result = run_react_agent(
    "I'm in Mumbai and want to fly to London. What's the weather like there, "
    "and what will the flight cost in Indian Rupees?"
)

TASK: I'm in Mumbai and want to fly to London. What's the weather like there, and what will the flight cost in Indian Rupees?
[Step 1] ACTION      → get_weather({'city': 'London'})
[Step 1] OBSERVATION → {'temp': 15, 'condition': 'rainy', 'rain_chance': 75}
[Step 1] ACTION      → get_flight_info({'origin': 'Mumbai', 'destination': 'London'})
[Step 1] OBSERVATION → {'route': 'Mumbai → London', 'duration_hours': 9, 'cheapest_fare_usd': 620, 'airline': 'SkyWings'}
[Step 2] ACTION      → convert_currency({'to_currency': 'INR', 'amount': 620, 'from_currency': 'USD'})
[Step 2] OBSERVATION → {'from': '620 USD', 'to': '51770.0 INR'}

[Step 3] THOUGHT → Final answer (no more tools needed)
[Step 3] ANSWER  → The weather in London is currently rainy with a temperature of 15°C and a 75% chance of rain.

Regarding your travel, a flight from Mumbai to London with SkyWings takes approximately 9 hours. The cheapest fare is 620 USD, which is approximately 51,770 INR.

📊 Total steps: 3 | Total tokens us

---
## Demo 2: Multi-Step Agent — Trip Cost Calculator

A realistic multi-step task: find the cheapest flight, look up hotel prices,  
and calculate total trip cost in the user's currency.

In [12]:
import random

def search_flights(origin: str, destination: str, date: str) -> dict:
    options = [
        {'flight_id': 'SK201', 'airline': 'SkyWings',  'price_usd': 580, 'duration_h': 9},
        {'flight_id': 'AE445', 'airline': 'AeroEast',  'price_usd': 640, 'duration_h': 11},
        {'flight_id': 'SW900', 'airline': 'SwiftAir',  'price_usd': 520, 'duration_h': 10},
    ]
    return {'route': f'{origin}→{destination}', 'date': date, 'options': options}

def search_hotels(city: str, checkin: str, checkout: str) -> dict:
    options = [
        {'hotel_id': 'H01', 'name': 'CityInn',     'price_per_night_usd': 85,  'rating': 3.8},
        {'hotel_id': 'H02', 'name': 'ParkView',    'price_per_night_usd': 140, 'rating': 4.3},
        {'hotel_id': 'H03', 'name': 'BudgetStay',  'price_per_night_usd': 55,  'rating': 3.2},
    ]
    return {'city': city, 'checkin': checkin, 'checkout': checkout, 'options': options}

def calculate_total_cost(flight_price_usd: float, hotel_price_per_night_usd: float, nights: int) -> dict:
    hotel_total = hotel_price_per_night_usd * nights
    grand_total = flight_price_usd + hotel_total
    inr_rate    = 83.5
    return {
        'flight_usd':     flight_price_usd,
        'hotel_usd':      hotel_total,
        'total_usd':      grand_total,
        'total_inr':      round(grand_total * inr_rate, 0),
        'nights':         nights
    }

TRIP_TOOLS = {
    'search_flights':      search_flights,
    'search_hotels':       search_hotels,
    'calculate_total_cost': calculate_total_cost,
}

trip_tool_schema = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name='search_flights',
            description='Search available flights between two cities on a given date.',
            parameters=types.Schema(type=types.Type.OBJECT,
                properties={
                    'origin':      types.Schema(type=types.Type.STRING),
                    'destination': types.Schema(type=types.Type.STRING),
                    'date':        types.Schema(type=types.Type.STRING, description='YYYY-MM-DD'),
                }, required=['origin', 'destination', 'date'])
        ),
        types.FunctionDeclaration(
            name='search_hotels',
            description='Search available hotels in a city for given dates.',
            parameters=types.Schema(type=types.Type.OBJECT,
                properties={
                    'city':     types.Schema(type=types.Type.STRING),
                    'checkin':  types.Schema(type=types.Type.STRING, description='YYYY-MM-DD'),
                    'checkout': types.Schema(type=types.Type.STRING, description='YYYY-MM-DD'),
                }, required=['city', 'checkin', 'checkout'])
        ),
        types.FunctionDeclaration(
            name='calculate_total_cost',
            description='Calculate total trip cost given flight price, hotel nightly rate, and number of nights.',
            parameters=types.Schema(type=types.Type.OBJECT,
                properties={
                    'flight_price_usd':            types.Schema(type=types.Type.NUMBER),
                    'hotel_price_per_night_usd':   types.Schema(type=types.Type.NUMBER),
                    'nights':                      types.Schema(type=types.Type.INTEGER),
                }, required=['flight_price_usd', 'hotel_price_per_night_usd', 'nights'])
        ),
    ]
)

def run_trip_agent(task: str, max_steps: int = 10) -> str:
    conversation = [types.Content(role='user', parts=[types.Part(text=task)])]
    total_tokens = 0
    print('=' * 65)
    print(f'TASK: {task}')
    print('=' * 65)

    for step in range(1, max_steps + 1):
        response = client.models.generate_content(
            model=MODEL, contents=conversation,
            config=cfg(tools=[trip_tool_schema], temperature=0.0)
        )
        total_tokens += response.usage_metadata.total_token_count

        fcs = [
            p.function_call
            for p in response.candidates[0].content.parts
            if hasattr(p, 'function_call') and p.function_call
        ]

        if not fcs:
            final = get_text(response)
            print(f'\n[Step {step}] FINAL ANSWER:')
            print(final)
            print(f'\n📊 Steps: {step} | Tokens: {total_tokens}')
            return final

        results = []
        for fc in fcs:
            result = TRIP_TOOLS[fc.name](**dict(fc.args))
            results.append(result)
            print(f'[Step {step}] ACTION  → {fc.name}({dict(fc.args)})')
            print(f'[Step {step}] RESULT  → {json.dumps(result, indent=2)[:200]}...')
            print()

        conversation.append(response.candidates[0].content)  # preserves thought_signature
        conversation.append(types.Content(role='user', parts=[
            types.Part(function_response=types.FunctionResponse(name=fc.name, response={'result': r}))
            for fc, r in zip(fcs, results)
        ]))

    return 'Max steps reached.'

run_trip_agent(
    "I want to fly from Mumbai to Berlin on 2025-08-15 and stay for 3 nights. "
    "Find me the cheapest flight and a hotel with at least 4-star rating. "
    "What's the total cost in INR?"
)

TASK: I want to fly from Mumbai to Berlin on 2025-08-15 and stay for 3 nights. Find me the cheapest flight and a hotel with at least 4-star rating. What's the total cost in INR?
[Step 1] ACTION  → search_flights({'origin': 'Mumbai', 'date': '2025-08-15', 'destination': 'Berlin'})
[Step 1] RESULT  → {
  "route": "Mumbai\u2192Berlin",
  "date": "2025-08-15",
  "options": [
    {
      "flight_id": "SK201",
      "airline": "SkyWings",
      "price_usd": 580,
      "duration_h": 9
    },
    {
    ...

[Step 2] ACTION  → search_hotels({'checkin': '2025-08-15', 'checkout': '2025-08-18', 'city': 'Berlin'})
[Step 2] RESULT  → {
  "city": "Berlin",
  "checkin": "2025-08-15",
  "checkout": "2025-08-18",
  "options": [
    {
      "hotel_id": "H01",
      "name": "CityInn",
      "price_per_night_usd": 85,
      "rating": 3.8...

[Step 3] ACTION  → calculate_total_cost({'hotel_price_per_night_usd': 140, 'flight_price_usd': 520, 'nights': 3})
[Step 3] RESULT  → {
  "flight_usd": 520,
  "hotel_us

'For your trip from Mumbai to Berlin (2025-08-15 to 2025-08-18), here are the best options found:\n\n*   **Cheapest Flight:** SwiftAir (SW900) at **$520**.\n*   **4-Star Hotel:** ParkView at **$140 per night**.\n\nThe total cost for the flight and 3 nights at the hotel is **78,490 INR** (approximately $940 USD).'

---
## Demo 3: Agent Self-Correction — When Tools Fail

Real tools fail. A robust agent should recognize an error and try an alternative strategy.

In [13]:
call_count = {'search_flights': 0}

def search_flights_flaky(origin: str, destination: str, date: str) -> dict:
    """Fails on the first call to simulate a real API error."""
    call_count['search_flights'] += 1
    if call_count['search_flights'] == 1:
        raise ValueError('API_ERROR: Flight search service temporarily unavailable. Try a nearby airport.')
    return {
        'route': f'{origin}→{destination}',
        'date': date,
        'options': [{'flight_id': 'BK99', 'airline': 'BackupAir', 'price_usd': 610, 'duration_h': 10}]
    }

def run_agent_with_error_handling(task: str, max_steps: int = 10):
    conversation = [
        types.Content(role='user', parts=[types.Part(text=
            'You are a helpful travel agent. If a tool returns an error, '
            'reason about it and try an alternative approach. ' + task
        )])
    ]
    print('=' * 65)
    print(f'TASK: {task}')
    print('=' * 65)

    for step in range(1, max_steps + 1):
        response = client.models.generate_content(
            model=MODEL, contents=conversation,
            config=cfg(tools=[trip_tool_schema], temperature=0.2)
        )
        fcs = [
            p.function_call
            for p in response.candidates[0].content.parts
            if hasattr(p, 'function_call') and p.function_call
        ]

        if not fcs:
            print(f'\n[Step {step}] FINAL ANSWER:\n{get_text(response)}')
            return

        tool_responses = []
        for fc in fcs:
            print(f'[Step {step}] ACTION → {fc.name}({dict(fc.args)})')
            try:
                if fc.name == 'search_flights':
                    result = search_flights_flaky(**dict(fc.args))
                else:
                    result = TRIP_TOOLS[fc.name](**dict(fc.args))
                print(f'[Step {step}] OK     → {result}')
                tool_responses.append({'name': fc.name, 'response': {'result': result}})
            except Exception as e:
                error_msg = str(e)
                print(f'[Step {step}] ERROR  → {error_msg}')
                print(f'           Agent will see this error and decide what to do next...')
                tool_responses.append({'name': fc.name, 'response': {'error': error_msg}})

        # KEY: use original content to preserve thought_signature
        conversation.append(response.candidates[0].content)
        conversation.append(types.Content(role='user', parts=[
            types.Part(function_response=types.FunctionResponse(
                name=tr['name'], response=tr['response']
            ))
            for tr in tool_responses
        ]))
        print()

run_agent_with_error_handling(
    "Find a flight from Mumbai to Berlin on 2025-08-20. Stay for 2 nights."
)

TASK: Find a flight from Mumbai to Berlin on 2025-08-20. Stay for 2 nights.
[Step 1] ACTION → search_flights({'origin': 'Mumbai', 'destination': 'Berlin', 'date': '2025-08-20'})
[Step 1] ERROR  → API_ERROR: Flight search service temporarily unavailable. Try a nearby airport.
           Agent will see this error and decide what to do next...

[Step 2] ACTION → search_flights({'destination': 'Berlin', 'date': '2025-08-20', 'origin': 'Pune'})
[Step 2] OK     → {'route': 'Pune→Berlin', 'date': '2025-08-20', 'options': [{'flight_id': 'BK99', 'airline': 'BackupAir', 'price_usd': 610, 'duration_h': 10}]}

[Step 3] ACTION → search_hotels({'checkin': '2025-08-20', 'city': 'Berlin', 'checkout': '2025-08-22'})
[Step 3] OK     → {'city': 'Berlin', 'checkin': '2025-08-20', 'checkout': '2025-08-22', 'options': [{'hotel_id': 'H01', 'name': 'CityInn', 'price_per_night_usd': 85, 'rating': 3.8}, {'hotel_id': 'H02', 'name': 'ParkView', 'price_per_night_usd': 140, 'rating': 4.3}, {'hotel_id': 'H03', 'name

---
## Demo 4: Chain vs. Agent — When to Use Which

An agent is powerful but expensive. Sometimes a fixed **prompt chain** is the right tool:  
faster, cheaper, fully deterministic. Run the same task both ways and compare.

In [14]:
TASK = 'Find a flight from Delhi to Paris on 2025-09-10 and a hotel for 4 nights. Summarize the cheapest options.'

# ── Approach A: Fixed 3-step prompt chain ──────────────────────
t0 = time.time()

# Step 1: hardcoded flight search
flights = search_flights('Delhi', 'Paris', '2025-09-10')
# Step 2: hardcoded hotel search
hotels  = search_hotels('Paris', '2025-09-10', '2025-09-14')
# Step 3: ask LLM to summarize
chain_resp = client.models.generate_content(
    model=MODEL,
    contents=f'Summarize the cheapest flight and hotel options from this data:\nFlights: {flights}\nHotels: {hotels}',
    config=cfg(temperature=0.0)
)
chain_time   = round(time.time() - t0, 2)
chain_tokens = chain_resp.usage_metadata.total_token_count

print('APPROACH A — Fixed Chain')
print(get_text(chain_resp))
print(f'\n⏱  Time: {chain_time}s | 🪙 Tokens: {chain_tokens}')

print('\n' + '─' * 65 + '\n')

# ── Approach B: ReAct agent ────────────────────────────────────
t0 = time.time()
run_trip_agent(TASK)
agent_time = round(time.time() - t0, 2)
print(f'⏱  Total agent time: {agent_time}s')

print('\n' + '=' * 65)
print('SUMMARY')
print('=' * 65)
print(f'Chain  : {chain_time}s, {chain_tokens} tokens — fast, deterministic, inflexible')
print(f'Agent  : {agent_time}s — flexible, handles ambiguity, more expensive')
print('\nRule of thumb: use a chain when the steps are known in advance;')
print('use an agent when the steps depend on intermediate results or user intent.')

APPROACH A — Fixed Chain
Here is the summary of the cheapest travel options for your trip to Paris (Sept 10–14, 2025):

*   **Cheapest Flight:** SwiftAir (Flight SW900) at **$520**.
*   **Cheapest Hotel:** BudgetStay at **$55 per night** ($220 total for 4 nights).

**Total estimated cost for the cheapest options: $740.**

⏱  Time: 0.87s | 🪙 Tokens: 408

─────────────────────────────────────────────────────────────────

TASK: Find a flight from Delhi to Paris on 2025-09-10 and a hotel for 4 nights. Summarize the cheapest options.
[Step 1] ACTION  → search_flights({'origin': 'Delhi', 'date': '2025-09-10', 'destination': 'Paris'})
[Step 1] RESULT  → {
  "route": "Delhi\u2192Paris",
  "date": "2025-09-10",
  "options": [
    {
      "flight_id": "SK201",
      "airline": "SkyWings",
      "price_usd": 580,
      "duration_h": 9
    },
    {
      ...

[Step 1] ACTION  → search_hotels({'checkin': '2025-09-10', 'city': 'Paris', 'checkout': '2025-09-14'})
[Step 1] RESULT  → {
  "city": "Paris

In [15]:
# ✏️  Give the agent a multi-step task of your own.
#    The agent has access to: get_weather, convert_currency, get_flight_info
#    Try to craft a question that requires at least 2 tool calls.

my_task = "TODO: write a multi-step travel question here"
run_react_agent(my_task)

TASK: TODO: write a multi-step travel question here

[Step 1] THOUGHT → Final answer (no more tools needed)
[Step 1] ANSWER  → I'm ready to help! Please go ahead and provide your multi-step travel question. 

For example, you could ask something like:
*   "What is the weather like in Tokyo right now, and how much would 500 USD be in Japanese Yen?"
*   "Are there any flights from New York to London, and what is the current weather in London?"

Just let me know what you need!

📊 Total steps: 1 | Total tokens used: 309


'I\'m ready to help! Please go ahead and provide your multi-step travel question. \n\nFor example, you could ask something like:\n*   "What is the weather like in Tokyo right now, and how much would 500 USD be in Japanese Yen?"\n*   "Are there any flights from New York to London, and what is the current weather in London?"\n\nJust let me know what you need!'

---
## Key Takeaways

| Concept | What it means |
|---|---|
| ReAct loop | Agent alternates Thought (reason) → Action (tool) → Observation until done |
| Agent = while-loop | No framework magic — just a loop that stops when no tool is called |
| Self-correction | Pass tool errors back to the LLM; it reasons about failures and retries |
| max_steps guard | Always cap the loop — prevents runaway agents burning tokens |
| Token growth | Each step adds to the context; long agents get expensive fast |
| Chain vs Agent | Fixed chain = faster/cheaper; agent = flexible but costly. Match to task. |